# Pipeline 3: Summarization

1. Introduction to the encoder-decoder architecture — how it differs from decoder-only (separate reading and generation steps)
2. Load the pipeline — model choice and why (`facebook/bart-large-cnn`), token setup
3. Basic example — single text, default parameters
4. Controlling output length — `min_length` and `max_length`, what happens when you push the limits
5. Abstractive vs extractive — what BART actually does vs a naive copy-paste approach

# Pipeline 3: Summarization

Encoder-decoder models (also called sequence-to-sequence models) use both parts of the Transformer architecture.
Given a long text as prompt, the model creates concise summaries of it, keeping the most important information.

At each stage, the attention layers of the encoder can access all the words in the initial sentence, whereas the attention layers of the decoder can only access the words positioned before a given word in the input.

In [1]:
from transformers import pipeline

/home/v/Desktop/Projects/001-transformers-course/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the environment variables

Load dotenv to be able to access the HuggingFace API token.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(".secrets")
hf_token = os.getenv("HF_TOKEN")

## Load the pipeline

We use `facebook/bart-large-cnn` — BART is a transformer encoder-encoder (seq2seq) model with a bidirectional (BERT-like) encoder and an autoregressive (GPT-like) decoder.


In [3]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cuda:0


## Basic example

`truncation=True` means that the model will not generate new tokens if the input text is too long. there is a limit of 1024 tokens.

In [4]:
ARTICLE = """ Días antes de que dos estudiantes de posgrado de la Universidad del Sur de Florida desaparecieran el mes pasado, un compañero de habitación de uno de ellos supuestamente le hizo una pregunta inusual al chatbot de inteligencia artificial ChatGPT.

"¿Qué pasa si a un humano lo ponen (sic) en una bolsa de basura negra y lo tiran en un contenedor?", preguntó Hisham Abugharbieh el 13 de abril, según una declaración jurada presentada por fiscales de Florida.

ChatGPT respondió que sonaba peligroso, indica el documento, y Abugharbieh hizo entonces otra pregunta: "¿Cómo lo descubrirían?".

Esas supuestas entradas en ChatGPT, incluidas en documentos judiciales que acusan a Abugharbieh de dos cargos de homicidio premeditado, son solo el ejemplo más reciente de investigadores que utilizan historiales de chats con IA como evidencia en investigaciones criminales. Una conversación con ChatGPT también se utilizó en el caso de incendio provocado durante los incendios forestales de Los Ángeles, y una conversación con la IA de Snapchat fue una prueba clave en un juicio por asesinato en Virginia en 2024.
 Para los investigadores, estos registros de chat pueden ofrecer información valiosa sobre el estado mental y el posible motivo de un sospechoso.

"Creo que cualquier comunicación con chatbots de IA es como un tesoro para las agencias de seguridad", dijo Ilia Kolochenko, experto en ciberseguridad y abogado en Washington, DC. "(Los sospechosos) creen que sus interacciones con la IA permanecerán confidenciales o, al menos, no se revelarán ni se descubrirán, por lo que con frecuencia hacen preguntas muy directas y explícitas".

Los casos criminales subrayan el creciente uso de los chatbots de IA para obtener consejos personales y la falta de protecciones de privacidad para esas conversaciones. Aunque los chatbots de IA se han convertido rápidamente en una fuente habitual para asesoría legal, diagnósticos médicos y terapia, esas conversaciones no están protegidas legalmente como lo estarían con un abogado, médico o terapeuta con licencia.

El CEO de OpenAI, Sam Altman, ha señalado que esta falta de privacidad es un "gran problema".

"La gente habla de las cosas más personales de su vida con ChatGPT", dijo Altman en julio pasado en un pódcast con el comediante Theo Von. "La gente lo usa, especialmente los jóvenes, como terapeuta, como coach de vida, para problemas de pareja. '¿Qué debería hacer?'".

"Y ahora mismo, si hablas con un terapeuta, un abogado o un médico sobre esos problemas, existe un privilegio legal. Hay confidencialidad médico-paciente, confidencialidad legal, lo que sea. Y todavía no hemos resuelto eso cuando hablas con ChatGPT. Así que si hablas con ChatGPT sobre temas muy sensibles y luego hay una demanda o algo así, podríamos estar obligados a entregar esa información".

Varios expertos legales consultados por CNN coincidieron con ese análisis y señalaron que no existe una expectativa de privacidad en las aplicaciones de chat con IA.
"""

print(summarizer(ARTICLE, max_length=130, min_length=30, do_sample=False, truncation=True))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'summary_text': 'Two estudiantes de Florida desaparecieran el mes pasado. Un compañero de habitación de one of them supuestamente le hizo una pregunta inusual al ChatGPT. Para los investigadores, estos registros de chat pueden ofrecer información valiosa sobre el estado mental.'}]


## Pushing the limits

`min_length` and `max_length` can control the output of the summary.

Limiting to much the summary length can result in loss of important information. It is important to strike a balance between brevity and retaining key details.

In [5]:
print(summarizer(ARTICLE, max_length=30, min_length=10, do_sample=False, truncation=True))

[{'summary_text': 'Two estudiantes de Florida desaparecieran el mes pasado. Un compañero de habitación'}]


In [6]:
print(summarizer(ARTICLE, max_length=230, min_length=130, do_sample=False, truncation=True))

[{'summary_text': 'Two estudiantes de Florida desaparecieran el mes pasado. Un compañero de habitación de one of them supuestamente le hizo una pregunta inusual al ChatGPT. Para los investigadores, estos registros de chat pueden ofrecer información valiosa sobre el estado mental y el posible motivo de un sospechoso. El CEO de OpenAI, Sam Altman, ha señalado que esta falta de privacidad es un "gran problema"'}]


## Abstractive vs extractive

Extractive approaches only choose the relevant text from the original document to create the summary, while abstractive approaches generate new text that captures the meaning of the original document.

In [7]:
import re

from collections import Counter

def extractive_summary(text, n=3):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    words = re.findall(r'\b\w+\b', text.lower())

    freq = Counter(words)
    stop = {'que', 'de', 'la', 'el', 'en', 'un', 'y', 'a', 'los', 'se', 'con', 'por', 'su', 'the', 'a', 'of', 'in'}
    for w in stop: freq.pop(w, None)

    scores = [(sum(freq.get(w, 0) for w in re.findall(r'\b\w+\b', s.lower())) / len(s.split()), s)
              for s in sentences]
    return ' '.join(s for _, s in sorted(scores, reverse=True)[:n])

print("=== ABSTRACTIVE (BART) ===")
print(summarizer(ARTICLE, max_length=130, min_length=30, do_sample=False, truncation=True)[0]['summary_text'])

print("\n=== EXTRACTIVE (frequency of words) ===")
print(extractive_summary(ARTICLE))

=== ABSTRACTIVE (BART) ===
Two estudiantes de Florida desaparecieran el mes pasado. Un compañero de habitación de one of them supuestamente le hizo una pregunta inusual al ChatGPT. Para los investigadores, estos registros de chat pueden ofrecer información valiosa sobre el estado mental.

=== EXTRACTIVE (frequency of words) ===
Hay confidencialidad médico-paciente, confidencialidad legal, lo que sea. "La gente lo usa, especialmente los jóvenes, como terapeuta, como coach de vida, para problemas de pareja. Aunque los chatbots de IA se han convertido rápidamente en una fuente habitual para asesoría legal, diagnósticos médicos y terapia, esas conversaciones no están protegidas legalmente como lo estarían con un abogado, médico o terapeuta con licencia.
